In [ ]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion


In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)
from sdv.evaluation.single_table import evaluate_quality

# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------
metro_interstate_traffic_volume = fetch_ucirepo(id=492)

X = metro_interstate_traffic_volume.data.features.copy()
y = metro_interstate_traffic_volume.data.targets.copy()

# Drop date_time before any preprocessing or generator training.
X = X.drop(columns=['date_time'], errors='ignore')

print(metro_interstate_traffic_volume.metadata)
print(metro_interstate_traffic_volume.variables)

if isinstance(y, pd.DataFrame):
    if 'traffic_volume' in y.columns:
        target_col = 'traffic_volume'
    else:
        target_col = y.columns[0]
    y_series = y[target_col]
else:
    target_col = 'traffic_volume'
    y_series = pd.Series(y, name=target_col)

In [ ]:
# ----------------------------------------------------
# Preprocess features before synthetic data generation
# ----------------------------------------------------
# 1. Drop the date_time column from the dataset
# 2. Keep raw columns for TabDDPM
# 3. One-hot encode categoricals for WGAN / SDV / downstream ML
# ----------------------------------------------------

# Drop date_time (hour of data collected in local CST time).
X = X.drop(columns=['date_time'], errors='ignore')

# Raw tabular features for TabDDPM (expects categorical column names, not dummies).
X_ctab = X.copy()
for col in ['temp', 'rain_1h', 'snow_1h', 'clouds_all']:
    X_ctab[col] = pd.to_numeric(X_ctab[col], errors='coerce')
metro_data_ctab = pd.concat([X_ctab, y_series.reset_index(drop=True)], axis=1)
metro_data_ctab[target_col] = pd.to_numeric(metro_data_ctab[target_col], errors='coerce').fillna(0)
for col in ['holiday', 'weather_main', 'weather_description']:
    metro_data_ctab[col] = metro_data_ctab[col].fillna('None').astype(str)

# One-hot encode categorical columns for other generators.
X = pd.get_dummies(X, drop_first=True)
X = X.fillna(0)

metro_data = pd.concat([X, y_series.reset_index(drop=True)], axis=1)
metro_data = metro_data.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float64)

print(f'Target variable: {target_col}')
print(f'date_time in features: {"date_time" in metro_data.columns}')
print(f'Encoded dataset shape: {metro_data.shape}')
print(f'TabDDPM raw dataset shape: {metro_data_ctab.shape}')

In [ ]:
# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000          # random real samples drawn from full dataset
TEST_SIZE = 0.2           # 20% holdout for unseen TSTR evaluation
SEED = 42

# Speed controls (set FAST_MODE=False for full paper epochs)
FAST_MODE = True
DEV_MODE = False
RUN_QUALITY_EVAL = True

N_SYNTH_SAMPLES = 1000

_epoch_fast = 5 if FAST_MODE else None
TabDDPM_EPOCHS = _epoch_fast if FAST_MODE else 150
WGAN_EPOCHS = (10 if FAST_MODE else 100)  # WGAN needs more epochs on high-dim one-hot data
SDV_EPOCHS = _epoch_fast if FAST_MODE else 300

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

ALL_GENERATORS = [
    'CTGAN', 'CopulaGAN', 'TVAE', 'GaussianCopula', 'TabDDPM', 'ForestDiffusion'
]
GENERATORS_TO_EVAL = ALL_GENERATORS

# Randomly select 1000 samples from the full preprocessed dataset.
_sample_idx = metro_data.sample(n=N_SAMPLES, random_state=SEED).index
metro_data = metro_data.loc[_sample_idx].reset_index(drop=True)
metro_data_ctab = metro_data_ctab.loc[_sample_idx].reset_index(drop=True)

# 80% for generator training, 20% held out unseen for TSTR evaluation.
train_real, test_real = train_test_split(
    metro_data,
    test_size=TEST_SIZE,
    random_state=SEED,
)
train_real_ctab, test_real_ctab = train_test_split(
    metro_data_ctab,
    test_size=TEST_SIZE,
    random_state=SEED,
)
train_real = train_real.reset_index(drop=True)
test_real = test_real.reset_index(drop=True)
train_real_ctab = train_real_ctab.reset_index(drop=True)
test_real_ctab = test_real_ctab.reset_index(drop=True)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(train_real)
train_metadata = metadata

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


def align_to_train_schema(df, reference_df, label_col):
    """Map raw or mixed-type rows to the one-hot numeric schema used by train_real."""
    df = df.copy()
    y = pd.to_numeric(df[label_col], errors='coerce').fillna(0)
    X = df.drop(columns=[label_col], errors='ignore')
    X_ref = reference_df.drop(columns=[label_col], errors='ignore')

    if X.select_dtypes(include=['object', 'string', 'category']).shape[1] > 0:
        X = pd.get_dummies(X, drop_first=True)

    X = X.reindex(columns=X_ref.columns, fill_value=0)
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float64)

    out = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)
    out.columns = reference_df.columns
    return out


print(f'Random subsample: {metro_data.shape}')
print(f'Generator training set (80%): {train_real.shape}')
print(f'Holdout test set (20%, unseen): {test_real.shape}')
print(f'DEV_MODE: {DEV_MODE} | FAST_MODE: {FAST_MODE} | quality eval: {RUN_QUALITY_EVAL}')
print(f'Generators enabled: {GENERATORS_TO_EVAL}')
print(f'Synthetic samples per generator: {N_SYNTH_SAMPLES}')


In [ ]:
# ---------------------------------------------------
# SINGLE RUN — setup + TabDDPM
# ---------------------------------------------------
seed = SEED
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Train generators on 80% of the 1000-sample subset only.

# TabDDPM — train on raw categoricals (not one-hot encoded)
# Use holiday + weather_main only (weather_description is too high-cardinality
# for 800-row train split and triggers Cond/log errors in TabDDPM+).

if 'TabDDPM' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training TabDDPM...')
        synthetic_tabddpm = train_tabddpm(
            train_real,
            target_col=target_col,
            categorical_columns=['holiday', 'weather_main'],
            n_samples=N_SYNTH_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['TabDDPM'] = synthetic_tabddpm.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_tabddpm,
                metadata=train_metadata,
            )
            scores['TabDDPM'] = quality.get_score()
            print('TabDDPM:', round(scores['TabDDPM'], 4))
        else:
            print('TabDDPM: trained (quality eval skipped)')
    except Exception as e:
        print('TabDDPM Failed:', e)
        traceback.print_exc()
else:
    print('TabDDPM: skipped (not in GENERATORS_TO_EVAL)')


In [ ]:
# ForestDiffusion
if 'ForestDiffusion' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training ForestDiffusion...')
        synthetic_forestdiffusion = train_forestdiffusion(
            train_real,
            target_col=target_col,
            categorical_columns=[target_col],
            n_samples=N_SYNTH_SAMPLES,
            seed=seed,
        )
        synthetic_datasets['ForestDiffusion'] = synthetic_forestdiffusion.copy()
        print('ForestDiffusion: synthesis complete')
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_forestdiffusion,
                metadata=train_metadata,
            )
            scores['ForestDiffusion'] = quality.get_score()
            print('ForestDiffusion:', round(scores['ForestDiffusion'], 4))
        else:
            print('ForestDiffusion: trained (quality eval skipped)')
    except Exception as e:
        print('ForestDiffusion Failed (training/sampling):')
        traceback.print_exc()
    if 'ForestDiffusion' in synthetic_datasets and RUN_QUALITY_EVAL:
        pass
else:
    print('ForestDiffusion: skipped (not in GENERATORS_TO_EVAL)')


In [ ]:
# ---------------------------------------------------
# SDV Models (CTGAN, CopulaGAN, TVAE, GaussianCopula)
# ---------------------------------------------------
sdv_models = {
    'CTGAN': CTGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    'CopulaGAN': CopulaGANSynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    'TVAE': TVAESynthesizer(metadata=train_metadata, epochs=SDV_EPOCHS),
    'GaussianCopula': GaussianCopulaSynthesizer(metadata=train_metadata),
}

for model_name, model in sdv_models.items():
    if model_name not in GENERATORS_TO_EVAL:
        print(f'{model_name}: skipped (not in GENERATORS_TO_EVAL)')
        continue
    try:
        model.fit(train_real)
        synthetic_data = model.sample(N_SYNTH_SAMPLES)
        synthetic_data = align_to_train_schema(synthetic_data, train_real, target_col)
        synthetic_datasets[model_name] = synthetic_data.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_data,
                metadata=train_metadata
            )
            scores[model_name] = quality.get_score()
            print(f'{model_name}: {round(scores[model_name], 4)}')
        else:
            print(f'{model_name}: trained (quality eval skipped)')
    except Exception as e:
        print(f'{model_name} Failed: {e}')


In [ ]:
# Quality summary
quality_results = []
for gen_name in GENERATORS_TO_EVAL:
    quality_results.append({
        'Generator': gen_name,
        'Quality_Score': scores.get(gen_name, np.nan),
        'Status': 'Success' if gen_name in synthetic_datasets else 'Failed'
    })

quality_df = pd.DataFrame(quality_results).sort_values('Quality_Score', ascending=False)
display(quality_df)

In [ ]:
# ---------------------------------------------------
# Column Shapes Quality Report (per generator)
# ---------------------------------------------------
import warnings
warnings.filterwarnings('ignore')

from sdv.evaluation.single_table import QualityReport
import matplotlib.pyplot as plt
import seaborn as sns

# Generators to compare (only those with synthetic output available).
model_order = [m for m in ['TabDDPM', 'ForestDiffusion'] if m in synthetic_datasets]

if not model_order:
    print('No synthetic datasets available for column-shape comparison.')
else:
    real_eval = train_real.drop(columns=[target_col], errors='ignore').copy()

    metadata_eval = SingleTableMetadata()
    metadata_eval.detect_from_dataframe(real_eval)

    column_shape_details = {}

    for model_name in model_order:
        synth_eval = synthetic_datasets[model_name].drop(columns=[target_col], errors='ignore').copy()
        synth_eval = synth_eval[real_eval.columns]

        qr = QualityReport()
        qr.generate(
            real_data=real_eval,
            synthetic_data=synth_eval,
            metadata=metadata_eval.to_dict()
        )

        details = qr.get_details('Column Shapes').sort_values('Score', ascending=False)
        column_shape_details[model_name] = details

    fig, axes = plt.subplots(1, len(model_order), figsize=(9 * len(model_order), 6), sharey=True)
    if len(model_order) == 1:
        axes = [axes]

    for ax, model_name in zip(axes, model_order):
        details = column_shape_details[model_name]

        sns.barplot(
            x='Column',
            y='Score',
            data=details,
            palette='viridis',
            ax=ax
        )

        ax.set_title(f'{model_name}: Column Shapes Similarity')
        ax.tick_params(axis='x', rotation=90)
        ax.set_xlabel('Column')
        ax.set_ylabel('Score')

    plt.tight_layout()
    plt.show()

    combined_details = pd.concat(
        [df.assign(Model=name) for name, df in column_shape_details.items()],
        ignore_index=True
    )

    display(combined_details.head(20))


In [ ]:
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

regressors = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    'SVR_RBF': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoost': GradientBoostingRegressor(random_state=42),
}

print(f'Regression evaluation: {len(regressors)} models, {len(EVAL_SEEDS)} seeds, {len(GENERATORS_TO_EVAL)} generators')


In [ ]:
def evaluate_regression_models(train_df, test_df, label_col, models, test_size=0.2, seeds=None, use_holdout=False, schema_df=None):
    if seeds is None:
        seeds = EVAL_SEEDS
    if schema_df is None:
        schema_df = test_df if use_holdout else train_df

    train_df = align_to_train_schema(train_df, schema_df, label_col)
    test_df = align_to_train_schema(test_df, schema_df, label_col)
    results = []

    for name, model in models.items():
        r2_scores = []
        mse_scores = []
        rmse_scores = []
        mae_scores = []

        for seed in seeds:
            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            if not use_holdout:
                X_train, _, y_train, _ = train_test_split(
                    X_train, y_train, test_size=test_size, random_state=seed
                )
                _, X_test, _, y_test = train_test_split(
                    X_test, y_test, test_size=test_size, random_state=seed
                )

            reg = clone(model)
            if 'random_state' in reg.get_params():
                reg.set_params(random_state=seed)

            reg.fit(X_train, y_train)
            y_pred = reg.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            r2_scores.append(r2_score(y_test, y_pred))
            mse_scores.append(mse)
            rmse_scores.append(np.sqrt(mse))
            mae_scores.append(mean_absolute_error(y_test, y_pred))

        results.append({
            'Model': name,
            'R2 Mean': np.mean(r2_scores),
            'R2 Std': np.std(r2_scores),
            'MSE Mean': np.mean(mse_scores),
            'MSE Std': np.std(mse_scores),
            'RMSE Mean': np.mean(rmse_scores),
            'RMSE Std': np.std(rmse_scores),
            'MAE Mean': np.mean(mae_scores),
            'MAE Std': np.std(mae_scores),
            'R2 (Mean±Std)': f"{np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}",
            'MSE (Mean±Std)': f"{np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}",
            'RMSE (Mean±Std)': f"{np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}",
            'MAE (Mean±Std)': f"{np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by='R2 Mean', ascending=False)


In [ ]:
print('TRTR (Train Real, Test Real) — 80% train / 20% holdout')
trtr_results = evaluate_regression_models(
    train_df=train_real,
    test_df=test_real,
    label_col=target_col,
    models=regressors,
    seeds=EVAL_SEEDS,
    use_holdout=True,
    schema_df=train_real,
)
display(trtr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

all_comparisons = []
for synth_name in GENERATORS_TO_EVAL:
    if synth_name not in synthetic_datasets:
        print(f'Skipping {synth_name} - no synthetic dataset')
        continue

    print(f'{synth_name} - TSTR (train on synthetic, test on 20% holdout)')
    tstr_results = evaluate_regression_models(
        train_df=synthetic_datasets[synth_name],
        test_df=test_real,
        label_col=target_col,
        models=regressors,
        seeds=EVAL_SEEDS,
        use_holdout=True,
        schema_df=train_real,
    )
    display(tstr_results[['Model', 'R2 (Mean±Std)', 'MSE (Mean±Std)', 'RMSE (Mean±Std)', 'MAE (Mean±Std)']])

    comparison = trtr_results.merge(tstr_results, on='Model', suffixes=('_TRTR', '_TSTR'))
    comparison['R2_Drop'] = comparison['R2 Mean_TRTR'] - comparison['R2 Mean_TSTR']
    comparison['MSE_Increase'] = comparison['MSE Mean_TSTR'] - comparison['MSE Mean_TRTR']
    comparison['RMSE_Increase'] = comparison['RMSE Mean_TSTR'] - comparison['RMSE Mean_TRTR']
    comparison['MAE_Increase'] = comparison['MAE Mean_TSTR'] - comparison['MAE Mean_TRTR']
    comparison['Synthetic_Model'] = synth_name
    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)
summary = (
    combined_comparison
    .groupby('Synthetic_Model', as_index=False)[['R2_Drop', 'MSE_Increase', 'RMSE_Increase', 'MAE_Increase']]
    .mean()
    .sort_values('R2_Drop')
)

display(summary)


In [ ]:
output_file = 'TRTR_TSTR_results_regression.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    quality_df.to_excel(writer, sheet_name='Quality_Metrics', index=False)
    trtr_results.to_excel(writer, sheet_name='TRTR_Results', index=False)
    combined_comparison.to_excel(writer, sheet_name='All_Comparisons', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    for synth_name in GENERATORS_TO_EVAL:
        if synth_name in combined_comparison['Synthetic_Model'].values:
            synth_results = combined_comparison[combined_comparison['Synthetic_Model'] == synth_name]
            synth_results.to_excel(writer, sheet_name=synth_name[:31], index=False)

print(f'Results saved to: {output_file}')
